# ATM E-Journal - manual batch run and path load

Two things, both driven from `config/atm_ejournal.conf`:

1. **`run_batches(...)`** - take the unprocessed journal files, and for **each batch**:
   parse -> write one parquet -> insert that parquet into Greenplum -> record the files in
   `processed_files.csv` -> delete the parquet -> next batch.
2. **`load_parquet_path(path)`** - point it at a parquet file or a folder of parquet and it
   writes **the whole thing** into Greenplum. By default it skips the parquet it already
   loaded (tracked in `processed/loaded_parquet.csv`), so re-running inserts only what has
   not been inserted yet.

Parsing happens on this host and Spark is used only to read the parquet and write it to
Greenplum, so this notebook works on any PySpark/Python combination (Spark 3.1.3 with
Python 3.11 included).

## 1. Imports and configuration

In [ ]:
import os
import sys
import logging

import pandas as pd

# Make the project's modules importable (src/ next to this notebook's parent).
ETL_HOME = os.path.abspath(os.path.join(os.getcwd(), ".."))      # <- EDIT if moved
os.environ["ATM_ETL_HOME"] = ETL_HOME
sys.path.insert(0, os.path.join(ETL_HOME, "src"))

from config_loader import load_config
from file_registry import ProcessedFileRegistry, discover_files
from manual_load import (LoadedParquetRegistry, describe_state, find_parquet_targets,
                         load_parquet_path, print_reports, print_state, process_path,
                         run_batches, unprocessed_files)

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s", force=True)
logging.getLogger("py4j").setLevel(logging.WARNING)

CONFIG_PATH = os.path.join(ETL_HOME, "config", "atm_ejournal.conf")
ETL_NAME = "atm_ejournal"
cfg = load_config(CONFIG_PATH, ETL_NAME)

print("ETL home  :", ETL_HOME)
print("config    :", CONFIG_PATH)
print("input     :", cfg.path("input.INPUT_PATH"))
print("parquet   :", cfg.path("parquet.PARQUET_PATH"))
print("tracking  :", cfg.path("tracking.PROCESSED_FILES_CSV"))
print("greenplum :", cfg.get("greenplum.GREENPLUM_URL"), "->",
      f"{cfg.get('greenplum.GREENPLUM_SCHEMA')}.{cfg.get('greenplum.GREENPLUM_TABLE')}")
print("writer    :", cfg.get("greenplum.GREENPLUM_WRITE_FORMAT"),
      "| strategy:", cfg.get("greenplum.GREENPLUM_LOAD_STRATEGY"),
      "| mode:", cfg.get("greenplum.GREENPLUM_WRITE_MODE"))
print("parse     :", cfg.get("parser.PARSE_ENGINE"),
      "| workers:", cfg.get("parser.PARSE_WORKERS"))

## 2. What is waiting to be processed

One line per ATM folder, then the files that are not yet in `processed_files.csv`.

In [ ]:
# How many files one call will take, and where the tracking CSVs live.
state = print_state(cfg)

In [ ]:
# Per ATM folder, and the first files still to be processed.
from file_registry import ProcessedFileRegistry

registry = ProcessedFileRegistry(cfg.path("tracking.PROCESSED_FILES_CSV"),
                                 key_mode=str(cfg.get("tracking.FILE_KEY_MODE", "path")))
already = registry.load_keys(refresh=True)

per_atm, pending = {}, []
for item in discover_files(cfg.path("input.INPUT_PATH"),
                           patterns=cfg.get_list("input.FILE_PATTERN"),
                           folder_depth=cfg.get_int("input.ATM_FOLDER_DEPTH", 1),
                           min_file_age_seconds=cfg.get_int("input.MIN_FILE_AGE_SECONDS", 0)):
    per_atm[item.atm_no] = per_atm.get(item.atm_no, 0) + 1
    if item.key(registry.key_mode) not in already:
        pending.append(item.relative_path)

for atm, count in sorted(per_atm.items()):
    print(f"{atm:<24} {count:>6} journal file(s)")

print(f"\nStill to process : {len(pending)}")
for name in pending[:10]:
    print("   ", name)
if len(pending) > 10:
    print(f"    ... and {len(pending) - 10} more")

## 3. Batch-wise: process and insert into Greenplum

Each batch is parsed, written to its own parquet, inserted into Greenplum and only then
recorded in `processed_files.csv`; the parquet is deleted once the insert is confirmed.
A batch that fails leaves its files unprocessed and the run moves on to the next batch.

* `BATCH_SIZE` - files per batch (`None` uses `input.BATCH_SIZE` from the config)
* `MAX_BATCHES` - stop after this many batches (`None` = everything still pending)
* `KEEP_PARQUET` - keep each batch's parquet instead of deleting it after the insert

> **Only N files processed?** One call takes `BATCH_SIZE x MAX_BATCHES` files.
> `BATCH_SIZE = None` uses `input.BATCH_SIZE` from the configuration (check what section 2
> printed), and `MAX_BATCHES = 1` stops after the first batch. Set `MAX_BATCHES = None` to
> work through everything that is pending.


In [ ]:
# ---- EDIT ME -------------------------------------------------------------
BATCH_SIZE   = None        # e.g. 500; None -> input.BATCH_SIZE
MAX_BATCHES  = 1           # e.g. 1 for a careful first batch, None for everything
KEEP_PARQUET = False
# --------------------------------------------------------------------------

reports = run_batches(
    cfg,
    batch_size=BATCH_SIZE,
    max_batches=MAX_BATCHES,
    keep_parquet=KEEP_PARQUET,
    on_batch=lambda r: print(f"{r.batch_id}: {r.files} file(s) -> {r.records} record(s) "
                             f"-> {r.rows_loaded} row(s) in Greenplum "
                             f"[{r.status}] {r.error}"),
)

print()
print_reports(reports)

# Where the run recorded the files it finished (NOT the notebook's folder).
csv_path = cfg.path("tracking.PROCESSED_FILES_CSV")
print(f"\nprocessed_files.csv : {csv_path}")
if os.path.exists(csv_path):
    print("rows in it          :", sum(1 for _ in open(csv_path)) - 1)
else:
    print("rows in it          : the file does not exist - no batch completed")


In [ ]:
# Run the rest of the pending files in the same way (no batch limit).
# reports = run_batches(cfg, max_batches=None,
#                       on_batch=lambda r: print(r.batch_id, r.status, r.rows_loaded))
# print_reports(reports)

## 4. Insert an existing parquet path into Greenplum

`load_parquet_path(cfg, PATH)` writes **the entire** parquet at `PATH` into Greenplum.
`PATH` can be

* a single `.parquet` file - `/analyticsShare/.../all_atm_data.parquet`
* a Spark parquet directory - `parquet/batch_0001/`
* a folder holding several of either - each one is loaded in turn

`ONLY_NEW = True` skips the targets recorded in `processed/loaded_parquet.csv`, so running
this again on the same folder inserts only the parquet that was not inserted yet. Set it to
`False` to reload everything under the path.

It also appends the journals the parquet was built from to `processed_files.csv` (every row
carries `SOURCE_FILE_KEY` / `SOURCE_PATH`), so those files are not parsed and loaded again
by section 3. Pass `record_processed_files=False` to load a parquet without touching the
tracking.


In [ ]:
# ---- EDIT ME -------------------------------------------------------------
PATH = cfg.path("parquet.PARQUET_PATH")      # e.g. "/analyticsShare/.../output/"
ONLY_NEW = True                              # False = load everything under PATH again
# --------------------------------------------------------------------------

print("targets under", PATH)
for target in find_parquet_targets(PATH):
    print("   ", target)

loaded = LoadedParquetRegistry(os.path.join(
    os.path.dirname(cfg.path("tracking.PROCESSED_FILES_CSV")), "loaded_parquet.csv"))
print("\nalready inserted:", len(loaded.loaded_paths(refresh=True)))

In [ ]:
path_reports = load_parquet_path(
    cfg, PATH, only_new=ONLY_NEW,
    on_target=lambda r: print(f"{r.parquet_path}: {r.rows} row(s) -> "
                              f"{r.rows_loaded} loaded [{r.status}] {r.error}"),
)

print()
print_reports(path_reports)

## 5. Parse and load a whole folder of journals

`process_path(cfg, PATH)` is section 3 pointed at another directory and with no batch
limit: everything under `PATH` that is not in `processed_files.csv` is parsed and inserted,
batch by batch.

In [ ]:
# JOURNAL_PATH = "/analyticsShare/yushan/08.ATM_text_convertor/ATM_EJOURNALS"
# reports = process_path(cfg, JOURNAL_PATH, batch_size=500,
#                        on_batch=lambda r: print(r.batch_id, r.status, r.rows_loaded))
# print_reports(reports)

## 6. Check what landed

In [ ]:
csv_path = cfg.path("tracking.PROCESSED_FILES_CSV")
if os.path.exists(csv_path):
    tracked = pd.read_csv(csv_path)
    print(f"{len(tracked)} file(s) in {csv_path}")
    display(tracked.tail(10))
    display(tracked.groupby(["batch_id", "status"]).size().rename("files").reset_index().tail(10))

loaded_csv = os.path.join(os.path.dirname(csv_path), "loaded_parquet.csv")
if os.path.exists(loaded_csv):
    print()
    display(pd.read_csv(loaded_csv).tail(10))

In [ ]:
# Rows in the Greenplum table, per run and batch.
from greenplum_loader import GreenplumLoader
from spark_session import build_spark_session, stop_spark_session

spark = build_spark_session(cfg)
try:
    target = (spark.read.format("jdbc")
              .option("url", cfg.get("greenplum.GREENPLUM_URL"))
              .option("dbtable", f"{cfg.get('greenplum.GREENPLUM_SCHEMA')}."
                                 f"{cfg.get('greenplum.GREENPLUM_TABLE')}")
              .option("user", cfg.get("greenplum.GREENPLUM_USER"))
              .option("password", cfg.get("greenplum.GREENPLUM_PASSWORD"))
              .option("driver", cfg.get("greenplum.GREENPLUM_DRIVER"))
              .load())
    print("rows in target:", target.count())
    target.groupBy("ETL_RUN_ID", "BATCH_ID").count().orderBy("ETL_RUN_ID", "BATCH_ID").show(20)
    target.groupBy("ATM_NO", "STATUS").count().orderBy("ATM_NO").show(20)
finally:
    stop_spark_session(spark)

## Notes

* **Nothing is marked processed before Greenplum confirms the insert**, and the parquet of a
  batch is deleted only after `processed_files.csv` has been updated - so an interrupted run
  is simply re-run: finished batches are skipped, unfinished files are picked up again.
* **Re-running a file** appends its rows again with `GREENPLUM_LOAD_STRATEGY: append`. Set it
  to `delete_insert_by_source_file` in the config if a reload must replace the file's
  previous rows instead.
* **The greenplum-spark connector** is used instead of the JDBC writer by setting
  `GREENPLUM_WRITE_FORMAT: greenplum` in the config; its `server.port`, `segment.num`,
  `numWriteTasks`, `gpfdist.sessions` and `compression` come from
  `GREENPLUM_CONNECTOR_OPTIONS`.
* The scheduled equivalent of this notebook is `python3 src/run_etl.py` (same functions,
  plus logging, run summary and the Airflow e-mails).